In [1]:
import json
import numpy as np
import pandas as pd

from sklearn.preprocessing import MinMaxScaler

In [2]:
feature_df = pd.read_csv("candidate_features.csv")

with open("jd_profile.json","r") as f:
    jd = json.load(f)

feature_df.head()

,candidate_id,experience,num_skills,career_length,education_count,avg_skill_duration,total_endorsements,github,profile_complete,response_rate,...,notice,interview_rate,production_ai_skills,avg_job_duration,job_hopper,production_score,startup_score,research_score,skill_experience_ratio,quality_score
0,CAND_0000001,6.9,17,2,1,24.823529,292,9.2,86.9,0.34,...,60,0.71,6,41.000000,0,4,2,0,2.463768,37.45
1,CAND_0000002,12.5,9,4,1,20.333333,70,-1.0,78.7,0.29,...,60,0.62,2,37.250000,0,1,2,2,0.720000,32.64
2,CAND_0000003,1.1,6,1,2,17.666667,47,-1.0,31.9,0.46,...,150,0.86,1,13.000000,1,0,0,0,5.454545,29.18
3,CAND_0000004,3.8,10,3,2,13.400000,80,-1.0,28.5,0.26,...,120,0.35,1,14.666667,1,1,2,2,2.631579,17.55
4,CAND_0000005,11.0,6,4,1,24.500000,88,-1.0,84.6,0.37,...,30,0.74,0,32.500000,0,0,2,0,0.545455,35.67


In [3]:
numeric_columns = [

    "experience",

    "production_ai_skills",

    "production_score",

    "startup_score",

    "profile_complete",

    "response_rate",

    "saved",

    "interview_rate",

    "career_length",

    "github",

    "avg_job_duration"

]

scaler = MinMaxScaler()

feature_df[numeric_columns] = scaler.fit_transform(
    feature_df[numeric_columns]
)

In [4]:
feature_df["technical_score"] = (

    0.30 * feature_df["production_ai_skills"]

    +

    0.40 * feature_df["production_score"]

    +

    0.30 * feature_df["github"]

)

In [6]:
feature_df["career_score"] = (

    0.50 * feature_df["experience"]

    +

    0.30 * feature_df["career_length"]

    +

    0.20 * feature_df["avg_job_duration"]

)

In [7]:
feature_df["behavior_score"] = (

    0.35 * feature_df["profile_complete"]

    +

    0.25 * feature_df["response_rate"]

    +

    0.20 * feature_df["saved"]

    +

    0.20 * feature_df["interview_rate"]

)

In [8]:
feature_df["innovation_score"] = (

    feature_df["startup_score"]

)

In [9]:
feature_df["penalty"] = 0

feature_df.loc[
    feature_df["job_hopper"] == 1,
    "penalty"
] += 0.05

feature_df.loc[
    feature_df["research_score"] > 3,
    "penalty"
] += 0.05

feature_df.loc[
    feature_df["skill_experience_ratio"] > 6,
    "penalty"
] += 0.10

/tmp/ipykernel_19036/3710166872.py:3: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05
 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05
 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05
 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05
 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05
 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05
 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05
 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05
 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05
 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05 0.05
 0.05 0.05 0.05]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  feature_df.loc[


In [10]:
feature_df["recruiter_score"] = (

    0.40 * feature_df["technical_score"]

    +

    0.25 * feature_df["career_score"]

    +

    0.20 * feature_df["behavior_score"]

    +

    0.10 * feature_df["innovation_score"]

    -

    feature_df["penalty"]

)

In [11]:
feature_df["confidence"] = (

    feature_df["profile_complete"]

    *

    (1 - feature_df["penalty"])

)

In [12]:
feature_df["final_score"] = (

    0.85 * feature_df["recruiter_score"]

    +

    0.15 * feature_df["confidence"]

)

In [13]:
feature_df = feature_df.sort_values(

    "final_score",

    ascending=False

).reset_index(drop=True)

feature_df["rank"] = np.arange(
    1,
    len(feature_df)+1
)

feature_df.head(20)

,candidate_id,experience,num_skills,career_length,education_count,avg_skill_duration,total_endorsements,github,profile_complete,response_rate,...,quality_score,technical_score,behavior_score,career_score,innovation_score,penalty,recruiter_score,confidence,final_score,rank
0,CAND_0001610,0.142857,18,0.250,1,51.000000,480,0.509950,1.000000,0.609195,...,54.85,0.852985,0.816585,0.189484,0.8,0.0,0.631882,1.000000,0.687100,1
1,CAND_0001581,0.421429,14,0.250,1,30.000000,187,0.730100,0.864384,0.218391,...,40.54,0.790458,0.508830,0.357937,0.4,0.0,0.547434,0.864384,0.594976,2
2,CAND_0000981,0.385714,14,0.125,1,23.857143,131,0.886816,0.889041,0.586207,...,50.71,0.634616,0.638633,0.347024,0.6,0.0,0.528329,0.889041,0.582436,3
3,CAND_0000354,0.378571,13,0.250,1,28.769231,206,0.654229,0.657534,0.471264,...,46.36,0.724840,0.593128,0.325397,0.6,0.0,0.549911,0.657534,0.566054,4
4,CAND_0001575,0.385714,15,0.250,1,24.000000,309,0.000000,0.864384,0.563218,...,41.42,0.528571,0.592719,0.331746,1.0,0.0,0.512909,0.864384,0.565630,5
5,CAND_0001384,0.628571,14,0.375,2,17.500000,104,0.621891,0.745205,0.919540,...,49.58,0.549424,0.635882,0.505952,0.4,0.0,0.513434,0.745205,0.548200,6
6,CAND_0000227,0.471429,15,0.375,2,23.666667,219,0.569652,0.571233,0.344828,...,39.22,0.619467,0.462257,0.399256,1.0,0.0,0.540052,0.571233,0.544729,7
7,CAND_0000665,0.371429,10,0.250,2,28.300000,202,0.730100,0.821918,0.172414,...,44.52,0.624744,0.594710,0.320437,0.4,0.0,0.488949,0.821918,0.538894,8
8,CAND_0000001,0.421429,17,0.125,1,24.823529,292,0.126866,0.847945,0.344828,...,37.45,0.615203,0.515225,0.377381,0.4,0.0,0.483471,0.847945,0.538142,9
9,CAND_0001469,0.621429,9,0.500,1,24.555556,99,0.366915,0.857534,0.827586,...,47.67,0.392932,0.648651,0.514048,0.6,0.0,0.475415,0.857534,0.532733,10


In [14]:
contributions = feature_df[[
    "technical_score",
    "career_score",
    "behavior_score",
    "innovation_score",
    "penalty",
    "final_score"
]]

contributions.describe().T

,count,mean,std,min,25%,50%,75%,max
technical_score,1734.0,0.186970,0.135701,0.000000,0.080000,0.163742,0.251429,0.852985
career_score,1734.0,0.361095,0.195887,0.008333,0.205754,0.341741,0.503770,0.839815
behavior_score,1734.0,0.384690,0.123900,0.069394,0.294099,0.383301,0.470474,0.816585
innovation_score,1734.0,0.319839,0.162670,0.000000,0.200000,0.400000,0.400000,1.000000
penalty,1734.0,0.008622,0.029111,0.000000,0.000000,0.000000,0.000000,0.150000
final_score,1734.0,0.290228,0.100155,-0.037801,0.226687,0.289725,0.359912,0.687100


In [15]:
feature_df.to_csv(

    "candidate_scores.csv",

    index=False

)

print(feature_df.shape)

(1734, 33)
